# Data Preprocessing Pipeline

Runs the full reproducible pipeline: Raw Data -> Cleaning -> Integration -> Deduplication -> Symptom Standardization -> Synthetic Augmentation -> Feature Engineering -> Validation -> Final Dataset.

This notebook calls the SAME functions used by `scripts/prepare_data.py` (no duplicated logic), so the notebook and the CLI script always stay in sync.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if (pathlib.Path.cwd() / '..' / 'src').exists() else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts'))
import pandas as pd
pd.set_option('display.max_columns', 20)


In [2]:
from src.preprocessing.clean import (
    load_symptom_vocabulary, load_real_disease_symptom_sets, build_disease_symptom_pool
)

vocab_df = load_symptom_vocabulary()
real_df = load_real_disease_symptom_sets()
print('Symptom vocabulary size:', len(vocab_df))
print('Unique real disease-symptom combinations:', len(real_df))
print('Duplicate rows removed from raw source:', real_df.attrs.get('duplicate_rows_removed'))
real_df.head()

Symptom vocabulary size: 131
Unique real disease-symptom combinations: 304
Duplicate rows removed from raw source: 4616


,disease,symptoms
0,Fungal infection,"(dischromic_patches, itching, nodal_skin_erupt..."
1,Fungal infection,"(dischromic_patches, nodal_skin_eruptions, ski..."
2,Fungal infection,"(dischromic_patches, itching, nodal_skin_erupt..."
3,Fungal infection,"(dischromic_patches, itching, skin_rash)"
4,Fungal infection,"(itching, nodal_skin_eruptions, skin_rash)"


## Disease -> Validated Real Symptom Pool (sample)

In [3]:
pool = build_disease_symptom_pool(real_df)
for disease in list(pool)[:5]:
    print(disease, '->', pool[disease])

Fungal infection -> ['dischromic_patches', 'itching', 'nodal_skin_eruptions', 'skin_rash']
Allergy -> ['chills', 'continuous_sneezing', 'shivering', 'watering_from_eyes']
GERD -> ['acidity', 'chest_pain', 'cough', 'stomach_pain', 'ulcers_on_tongue', 'vomiting']
Chronic cholestasis -> ['abdominal_pain', 'itching', 'loss_of_appetite', 'nausea', 'vomiting', 'yellowing_of_eyes', 'yellowish_skin']
Drug Reaction -> ['burning_micturition', 'itching', 'skin_rash', 'spotting_urination', 'stomach_pain']


## Synthetic Augmentation (subset resampling of validated real symptoms only)

In [4]:
from src.preprocessing.augment import generate_synthetic_records
from src.utils.paths import RANDOM_SEED

synth_df = generate_synthetic_records(real_df, pool, target_per_disease=50, seed=RANDOM_SEED)
print('Synthetic records generated (demo target=50/disease):', len(synth_df))
synth_df.head()

Synthetic records generated (demo target=50/disease): 1290


,disease,symptoms
0,Fungal infection,"(itching, nodal_skin_eruptions)"
1,Fungal infection,"(dischromic_patches, nodal_skin_eruptions)"
2,Fungal infection,"(nodal_skin_eruptions, skin_rash)"
3,Fungal infection,"(itching, skin_rash)"
4,Fungal infection,"(dischromic_patches, itching)"


## Full Pipeline Execution

Runs the exact same `scripts/prepare_data.py` used to build the committed `data/processed/disease_dataset.csv` (target = 600 records/disease).

In [5]:
import prepare_data
prepare_data.main()

STEP 1/2: Loading & cleaning raw source data


Real unique disease-symptom combinations after dedup: 304
Duplicate rows removed from raw source: 4616
STEP 2/2: Synthetic augmentation for class balance & dataset size


Synthetic records generated: 9997



Saved final dataset -> C:\Users\Lenovo\OneDrive\Desktop\AI-Disease-Prediction\data\processed\disease_dataset.csv (10301 rows)
Saved symptom vocabulary -> C:\Users\Lenovo\OneDrive\Desktop\AI-Disease-Prediction\data\processed\symptom_vocabulary.json (131 symptoms)
Saved data dictionary -> C:\Users\Lenovo\OneDrive\Desktop\AI-Disease-Prediction\docs\data_dictionary.md

DATA PREPARATION SUMMARY
n_records: 10301
n_diseases: 41
n_symptoms: 131
n_real_records: 304
n_synthetic_records: 9997
class_imbalance_ratio_max_min: 54.55
min_class_count: 11
max_class_count: 600
issues_found_and_fixed: []
